In [1]:
import os
import glob
import re
import time
import code
import numpy as np
import anndata as ad
import pandas as pd
import anndata
import pyfaidx
import pybedtools
from pybedtools import BedTool
import anndata as ad
import scipy.sparse as sp_sparse
import pickle
import scanpy as sc
import muon.atac as ma  # 用于 TF-IDF
import networkx as nx
from matplotlib import rcParams
import sys

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2
sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/")


data_path = "/home/liyang/BioWuYan/MethodTest/Data/All2/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/original"

In [ ]:
adata_rna = ad.read_h5ad(data_path + "rna_origin.h5ad")

adata_atac = ad.read_h5ad(data_path + "atac_origin.h5ad")

jaspar_tf = pd.read_pickle(data_path + "jaspar_data.pkl")

chipseq_tf = pd.read_pickle(data_path + "tf_chip_seq_data.pkl")

print(jaspar_tf)

print(chipseq_tf)

print(set(jaspar_tf["name"]) & set(chipseq_tf["tf"]) & set(adata_rna.var_names))

In [ ]:
print(chipseq_tf.sort_values(by="signal", ascending=False))

print(len(set(chipseq_tf["tf"])))

print(len(set(jaspar_tf["name"])))

In [ ]:

chipseq_df = chipseq_tf[["chrom", "start", "end", "tf"]].copy()
chipseq_df.columns = ["chr", "start", "end", "name"]

tf_df = chipseq_df.copy()

standard_chroms = tuple([f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY'])

tf_df = tf_df[tf_df["chr"].isin(standard_chroms)].copy()
tf_df.drop_duplicates(inplace=True)
tf_df.sort_values(by=["name","chr", "start", "end"], inplace=True)
print(tf_df)

In [ ]:
tf_df = tf_df[tf_df["name"].isin(adata_rna.var_names)].copy()

print(tf_df)

In [ ]:
tf_df["name"].value_counts()

# RNA preprocess

## gene info

In [3]:
from src.data_preprocess import process_gene_info

adata_rna = ad.read_h5ad(data_path + "rna_origin.h5ad")

gtf_df = pd.read_pickle(data_path + "gene_info_data.pkl")

gtf_df = process_gene_info(gtf_df)

filter_gtf = gtf_df[gtf_df["gene_name"].isin(adata_rna.var_names)].copy()



## Filter and Process RNA

In [8]:
from src.data_preprocess import preprocess_adata_rna, filter_adata_rna

adata_rna = filter_adata_rna(adata_rna)

common_gene =sorted(list( set(filter_gtf["gene_name"]) & set(adata_rna.var_names) ))

adata_rna = adata_rna[:, common_gene].copy()

adata_rna = preprocess_adata_rna(adata_rna, num_high_variable_gene=2000)

/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:283: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


In [10]:
adata_rna = adata_rna[:, adata_rna.var['highly_variable']].copy()

filter_gtf = gtf_df[gtf_df["gene_name"].isin(adata_rna.var_names)].copy()

adata_rna.write_h5ad(output_path + "rna_processed.h5ad")

filter_gtf.to_pickle(output_path + "gene_info_filtered.pkl")

# ATAC preprocess

## DataRead

In [97]:
data_path =  "/home/liyang/BioWuYan/MethodTest/Data/All2/"

adata_atac = ad.read_h5ad(data_path + "atac_origin.h5ad")

tf_chip_seq_scenic = pd.read_parquet("/home/liyang/BioWuYan/MethodTest/Data/All/SCENIC_plus/combined_tf_peaks_robust.parquet")

print(tf_chip_seq_scenic.columns)

hic_data_df = pd.read_pickle(output_path + 'hic_data.pkl')

print(hic_data_df.columns)

gene_info = pd.read_pickle(output_path + "gene_info_filtered.pkl")

print(gene_info.columns)

Index(['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal_value',
       'p_value', 'q_value', 'peak', '10', '11', '12', '13', '14', '15', '16',
       '17', '18', '19', 'tf_name', 'cell_type', 'source_file_id'],
      dtype='object')
Index(['chr1', 'start1', 'end1', 'chr2', 'start2', 'end2', 'contact_count'], dtype='object')
Index(['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand',
       'frame', 'gene_id', 'gene_type', 'gene_name', 'level', 'tag',
       'transcript_id', 'transcript_type', 'transcript_name', 'exon_number',
       'exon_id', 'transcript_support_level', 'havana_transcript', 'hgnc_id',
       'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl'],
      dtype='object')


## Data Preprocess

In [11]:
from src.data_read import load_jaspar_folder_to_pfm

jaspar_folder_path = "/home/liyang/BioWuYan/MethodTest/Data/All/jaspar/"

pfm_dict, motif_names = load_jaspar_folder_to_pfm(jaspar_folder_path)

Find 879 motif file in /home/liyang/BioWuYan/MethodTest/Data/All/jaspar/
成功加载并解析了 828 个 motifs。


In [14]:
from src.data_preprocess import filter_atac_peaks_integrated, filter_atac_base

adata_atac = filter_atac_base(adata_atac)

df_res = filter_atac_peaks_integrated(adata_atac, tf_chip_seq_scenic, hic_data_df, gene_info, 
                                 distance_threshold=250000)

df_res.to_pickle(output_path + "atac_peak_filtering_results.pkl")

adata_atac = filter_atac_base(adata_atac)

gold_peaks = df_res[df_res['Tier_Code'] == 1]

filter_peak = sorted( list( set(gold_peaks["PeakID"]) & set(adata_atac.var_names) ) )

filter_atac = adata_atac[:, filter_peak].copy()

filter_atac.write_h5ad(output_path + "atac_filtered.h5ad")

['Lhx3',
 'TCF4',
 'ZNF213',
 'Arid5a',
 'Foxf1',
 'FOSL2::JUN',
 'POU1F1',
 'NR3C1',
 'Sox1',
 'Jun',
 'HOXC11',
 'HOXC4',
 'Six4',
 'Ahr::Arnt',
 'FOXO1::ELK1',
 'TBXT',
 'HOXD11',
 'IRF2',
 'Runx1',
 'HOXC13',
 'EHF',
 'SOX4',
 'CREM',
 'ZFP42',
 'OLIG3',
 'SOX8',
 'RARG',
 'Nfat5',
 'HINFP',
 'ATF7',
 'HOXD12',
 'Ptf1A',
 'KLF17',
 'Rarg',
 'ZEB1',
 'MYOD1',
 'Prdm15',
 'TEAD2',
 'Nobox',
 'ETV5::DRGX',
 'HMBOX1',
 'Ddit3::Cebpa',
 'HOXB2::ELK1',
 'UNCX',
 'HOXD9',
 'E2F4',
 'ZNF189',
 'PAX1',
 'BACH2',
 'DUX4',
 'HES2',
 'JUND',
 'FERD3L',
 'CDX1',
 'HOXA4',
 'Sox3',
 'POU4F1',
 'MYF5',
 'ETV6',
 'HOXB4',
 'CEBPG',
 'PKNOX2',
 'Mecom',
 'Zic3',
 'ZBTB24',
 'ZNF211',
 'IRF3',
 'OTX1',
 'Stat6',
 'FOXP3',
 'Ikzf3',
 'Zic2',
 'TEF',
 'Zfx',
 'NR2F1',
 'Mlxip',
 'ZNF677',
 'NR2F2',
 'TBX5',
 'NEUROG2',
 'Thap11',
 'RARA::RXRG',
 'TEAD3',
 'NKX6-2',
 'ZSCAN4',
 'Arid3a',
 'TFAP2C',
 'NR1H4::RXRA',
 'Bach1::Mafk',
 'ZBED1',
 'Foxn1',
 'MAX',
 'SMAD2',
 'Pparg::Rxra',
 'CEBPG',
 'Znf423'

In [6]:
adata_atac = ad.read_h5ad(output_path + "atac_processed.h5ad")

max_peaks = min(4500, adata_atac.n_vars)

adata_atac.layers['counts'] = adata_atac.X.copy()

print("--- 步骤 5: 运行下游分析 (TF-IDF, SVD, UMAP) ---")

# 1. TF-IDF 归一化 (等同于 Seurat::RunTFIDF)
ma.pp.tfidf(adata_atac, scale_factor=1e4)

sc.pp.highly_variable_genes(
    adata_atac,
    n_top_genes=max_peaks, # 选择所有 peak
    flavor='seurat_v3' # 使用 'seurat_v3' 算法
)

sc.tl.pca(adata_atac, n_comps=50, use_highly_variable=True, svd_solver='arpack')

n_dims_to_use = 29
n_comps_to_use = 30

if adata_atac.obsm['X_pca'].shape[1] < n_comps_to_use:
    sc.tl.pca(adata_atac, n_comps=n_comps_to_use, use_highly_variable=True, svd_solver='arpack')

adata_atac.obsm['X_lsi'] = adata_atac.obsm['X_pca'][:, 1:n_comps_to_use]

print(f"已创建 'X_lsi'，使用 SVD/PCA 成分 2 到 {n_comps_to_use} (共 {n_dims_to_use} 个)")

sc.pp.neighbors(adata_atac, n_neighbors=30, n_pcs=n_dims_to_use, use_rep='X_lsi')

sc.tl.umap(adata_atac)

sc.tl.leiden(adata_atac, resolution=1.0, key_added='leiden') 

adata_atac.layers['norm'] = adata_atac.X.copy()

adata_atac.X = adata_atac.layers['counts'].copy()

--- 步骤 5: 运行下游分析 (TF-IDF, SVD, UMAP) ---


/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:377: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)


已创建 'X_lsi'，使用 SVD/PCA 成分 2 到 30 (共 29 个)


/tmp/ipykernel_65160/1258740533.py:34: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_atac, resolution=1.0, key_added='leiden')


# TF-peak 

## JASPAR tf-peak

In [ ]:
from src.data_read import read_jaspar_folder

jaspar_folder_path = "/home/liyang/BioWuYan/MethodTest/Data/All/jaspar/"

genome_fasta_file = "/home/liyang/BioWuYan/MethodTest/Data/All/hg38.fa"

jaspar_data = read_jaspar_folder(jaspar_folder_path, genome_fasta_file, adata_atac)



In [16]:
from src.data_preprocess import filter_jaspar_tf

jaspar_tf_peak = ad.read_h5ad(output_path + "jaspar_data.h5ad")

jaspar_data_processed = filter_jaspar_tf(jaspar_tf_peak)

print(jaspar_data_processed)

/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。
  > 最终形状: (72563, 879)
AnnData object with n_obs × n_vars = 72563 × 879
    uns: 'description'


/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [11]:
if sp_sparse.issparse(jaspar_tf_peak.X):
    nan_mask = np.isnan(jaspar_tf_peak.X.data)
    if np.any(nan_mask):
        print(f"  > 发现了 {np.sum(nan_mask)} 个 NaN 值, 将其设为 0 ...")
        jaspar_tf_peak.X.data[nan_mask] = 0.0
else:
    if np.any(np.isnan(jaspar_tf_peak.X)):
        print(f"  > 发现了 NaN 值, 将其设为 0 ...")
        jaspar_tf_peak.X = np.nan_to_num(jaspar_tf_peak.X, nan=0.0)


score_threshold = 0
jaspar_tf_peak.X = (jaspar_tf_peak.X > score_threshold).astype(int)
jaspar_tf_peak.X = sp_sparse.csr_matrix(jaspar_tf_peak.X)

print(jaspar_tf_peak.X)


  (0, 7)	1
  (0, 9)	1
  (0, 10)	1
  (0, 11)	1
  (0, 12)	1
  (0, 15)	1
  (0, 18)	1
  (0, 19)	1
  (0, 22)	1
  (0, 24)	1
  (0, 27)	1
  (0, 29)	1
  (0, 30)	1
  (0, 33)	1
  (0, 34)	1
  (0, 37)	1
  (0, 40)	1
  (0, 44)	1
  (0, 48)	1
  (0, 49)	1
  (0, 50)	1
  (0, 54)	1
  (0, 56)	1
  (0, 57)	1
  (0, 58)	1
  :	:
  (72583, 848)	1
  (72583, 849)	1
  (72583, 850)	1
  (72583, 851)	1
  (72583, 852)	1
  (72583, 853)	1
  (72583, 854)	1
  (72583, 855)	1
  (72583, 856)	1
  (72583, 857)	1
  (72583, 859)	1
  (72583, 861)	1
  (72583, 863)	1
  (72583, 864)	1
  (72583, 865)	1
  (72583, 866)	1
  (72583, 867)	1
  (72583, 869)	1
  (72583, 871)	1
  (72583, 872)	1
  (72583, 873)	1
  (72583, 874)	1
  (72583, 875)	1
  (72583, 876)	1
  (72583, 878)	1


In [12]:
# ************* 1. Peak Filter (Rows) **************************

peak_counts = jaspar_tf_peak.X.getnnz(axis=1)
keep_peaks_mask = peak_counts > 0

print(f"\n步骤 1: 过滤 Peaks (行)")
print(f"  > 找到 {np.sum(keep_peaks_mask)} / {jaspar_tf_peak.n_obs} 个 peaks 至少有 1 个 TF 结合。")
jaspar_tf_peak = jaspar_tf_peak[keep_peaks_mask, :].copy()

# ************* 2. TF Filter (Columns) **************************

tf_counts = jaspar_tf_peak.X.getnnz(axis=0)
keep_tfs_mask = tf_counts > 0

print(f"\n步骤 2: 过滤 TFs (列)")
print(f"  > 找到 {np.sum(keep_tfs_mask)} / {jaspar_tf_peak.n_vars} 个 TFs 至少结合 1 个 peak。")

jaspar_tf_peak = jaspar_tf_peak[:, keep_tfs_mask].copy()

print(f"  > 最终形状: {jaspar_tf_peak.shape}")


步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。
  > 最终形状: (72563, 879)


/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


## ChIP-seq TF-peak

In [48]:
import pandas as pd

from src.data_preprocess import build_tf_peak_network

# 读取整个文件
tf_chip_seq_scenic = pd.read_parquet("/home/liyang/BioWuYan/MethodTest/Data/All/SCENIC_plus/combined_tf_peaks_robust.parquet")

tf_chip_df = tf_chip_seq_scenic[["chrom", "start", "end", "tf_name"]].copy()

tf_peak_network = build_tf_peak_network(adata_atac, tf_chip_df)

tf_peak_network.write_h5ad(output_path + "tf_peak_network.h5ad")

--- 1. 准备 ATAC Peak 数据 ---
未提供过滤列表，将使用 atac_adata 中所有的 Peaks。
ATAC Peaks 数量: 72569
--- 2. 准备 TF ChIP 数据 ---
TF ChIP Peaks 数量: 25217134
--- 3. 计算重叠 (构建 Network) ---
构建了包含 2765671 条边的 TF-Peak 网络。
--- 4. 转换为矩阵 (Matrix Construction) ---
--- 5. 封装为 AnnData ---
✅ 完成！生成的 AnnData 维度: (72569, 112) (Peaks x TFs)


In [49]:
len(set(tf_chip_df["tf_name"])) 

112

# Peak-Gene

## Hi-C data

In [51]:
hic_data_df = pd.read_pickle(output_path + 'hic_data.pkl')

filter_gtf = pd.read_pickle(output_path + "gene_info_filtered.pkl")

from src.data_preprocess import build_peak_gene_network

peak_gene_grn = build_peak_gene_network(adata_atac, hic_data_df, filter_gtf) 

peak_gene_grn.write_h5ad(output_path + "peak_gene_network.h5ad")

ImportError: cannot import name 'build_peak_gene_matrix' from 'src.data_preprocess' (/home/liyang/BioWuYan/MethodTest/dygmamba/src/data_preprocess.py)

## Potential

In [87]:
from src.data_preprocess import calculate_rp_250kb

adata_rp = calculate_rp_250kb(
    adata_atac=adata_atac, 
    adata_rna=adata_rna, 
    gene_info_df=gene_info, # 您的 DataFrame
    decay_dist=50000,       # 衰减参数：50kb (推荐)
    max_range=250000        # 硬性限制：250kb
)


--- 开始计算 RP Score (Decay=50000, Range=250000) ---
--- 1. 清洗 Gene Info ---


Chromosomes: 100%|██████████| 23/23 [00:00<00:00, 37.07it/s]

--- 3. 构建 AnnData 对象 ---
✅ 完成! 矩阵维度: (2000, 72569)
非零连接数: 50395


In [88]:
from src.data_preprocess import calculate_peak_peak_rp

adata_peak_rp = calculate_peak_peak_rp(adata_atac, 
                           decay_distance=50000, # 推荐 50kb
                           max_range=250000)


--- 计算 Peak-Peak RP (Range=250.0kb, Decay=50.0kb) ---


Chromosomes: 100%|██████████| 48/48 [00:00<00:00, 84.32it/s]


--- 构建全局稀疏矩阵 ---
--- 封装为 AnnData ---
✅ 完成! 矩阵维度: (72569, 72569)
Sparsity: 0.0321%


# TF-Gene

In [56]:
from src.data_preprocess import build_tf_gene_network_from_anndata

adata_tf_gene = build_tf_gene_network_from_anndata(tf_peak_network, peak_gene_grn)


--- 构建 TF-Gene 网络 (AnnData 矩阵乘法版) ---
TF矩阵 Peaks: 72569
Gene矩阵 Peaks: 72569
共有 Peaks: 72569
正在进行稀疏矩阵乘法...
正在封装结果...
✅ 构建完成! 矩阵维度: (112, 2000) (TFs x Genes)
共发现 223981 条潜在的 TF-Gene 调控关系。


# Convert to R

In [ ]:
import scipy.io as sio

adata_rna = ad.read_h5ad(data_path + "original_adata_rna_v2.h5ad")


adata_atac = ad.read_h5ad(data_path + "original_adata_atac_v2.h5ad")


atac_dir = data_path + "R/atac/"
os.makedirs(atac_dir, exist_ok=True)

cellinfo = adata_atac.obs
atacinfo = adata_atac.var

mtx = adata_atac.X

cellinfo.to_csv(atac_dir + "cellinfo.csv")
atacinfo.to_csv(atac_dir + "atacinfo.csv")

sio.mmwrite(atac_dir + "sparse.mtx", mtx)



rna_dir = data_path + "R/rna/"
os.makedirs(rna_dir, exist_ok=True)

cellinfo = adata_rna.obs
rnainfo = adata_rna.var

mtx = adata_rna.layers['counts']

cellinfo.to_csv(rna_dir + "cellinfo.csv")
rnainfo.to_csv(rna_dir + "rnainfo.csv")

sio.mmwrite(rna_dir + "sparse.mtx", mtx)

# Enviroment

source /home/liyang/miniconda3/etc/profile.d/conda.sh

conda activate /home/liyang/BioWuYan/conda_env/singlecellpreprocess/

In [132]:
from src.data_read import read_IDR_peaks_TFs


print("\n ********************---read chip-seq---***************************** \n")

data_path = "/home/liyang/BioWuYan/MethodTest/Data/All2/"

root_dir = data_path + "SCENIC_plus"

output_file = output_path + "combined_chip_seq.parquet"

tf_chip_seq_scenic = read_IDR_peaks_TFs(root_dir, output_file)


 ********************---read chip-seq---***************************** 

找到 309 个文件，使用【自适应模式】读取...


100%|██████████| 309/309 [00:54<00:00,  5.66it/s]



正在合并数据...
合并完成。前5行预览：
   chrom     start       end name  score strand  signal_value  p_value  \
0  chr10  73848806  73849564    .   1000      .     890.96251     -1.0   
1  chr10  79196719  79197439    .   1000      .     884.55247     -1.0   
2  chr10  80046693  80047460    .   1000      .     859.37243     -1.0   
3  chr10  79400349  79401068    .   1000      .     856.19252     -1.0   
4  chr10  79195312  79195956    .   1000      .     847.54568     -1.0   

   q_value  peak  ...        13         14   15        16        17  \
0  5.09063   398  ...  73849492  414.34388  343  73848847  73849487   
1  5.09063   342  ...  79197387  461.60624  296  79196799  79197323   
2  5.09063   375  ...  80047339  442.04534  278  80046749  80047366   
3  5.09063   315  ...  79401036  393.27076  270  79400386  79400928   
4  5.09063   312  ...  79195885  416.59550  259  79195384  79195897   

          18   19  tf_name  cell_type  source_file_id  
0  472.62535  359     CTCF       PC-3     ENCFF22

# Total

In [5]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"
adata_atac = ad.read_h5ad(output_path + "atac_origin.h5ad")
adata_rna_file = output_path + "rna_processed.h5ad"
adata_rna = ad.read_h5ad(adata_rna_file)
gene_info = pd.read_pickle(output_path + "gene_info_filtered.pkl")
decay_dist=50000      # 衰减参数：50kb (推荐)
max_range=250000        # 硬性限制：250kb
gene_info_df=gene_info

In [19]:
gene_df

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_type,...,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl,tss,gene_length
gene_name,,,,,,,,,,,,,,,,,,,,,
AAAS,chr12,HAVANA,gene,53307456,53324864,NaN,-,0,ENSG00000094914.15,protein_coding,...,,,HGNC:13666,OTTHUMG00000169729.12,,,,,53324864,17.408
AAGAB,chr15,HAVANA,gene,67200667,67255195,NaN,-,0,ENSG00000103591.14,protein_coding,...,,,HGNC:25662,OTTHUMG00000172246.4,,,,,67255195,54.528
AAR2,chr20,HAVANA,gene,36236131,36270918,NaN,+,0,ENSG00000131043.14,protein_coding,...,,,HGNC:15886,OTTHUMG00000032380.3,,,,,36236131,34.787
AASDHPPT,chr11,HAVANA,gene,106075501,106098708,NaN,+,0,ENSG00000149313.12,protein_coding,...,,,HGNC:14235,OTTHUMG00000166253.3,,,,,106075501,23.207
ABCB4,chr7,HAVANA,gene,87401696,87480435,NaN,-,0,ENSG00000005471.20,protein_coding,...,,,HGNC:45,OTTHUMG00000023396.20,,,,,87480435,78.739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSWIM1,chr20,HAVANA,gene,45881227,45885266,NaN,+,0,ENSG00000168612.6,protein_coding,...,,,HGNC:16155,OTTHUMG00000074023.2,,,,,45881227,4.039
ZSWIM8,chr10,HAVANA,gene,73785606,73801797,NaN,+,0,ENSG00000214655.11,protein_coding,...,,,HGNC:23528,OTTHUMG00000018486.5,,,,,73785606,16.191
ZWILCH,chr15,HAVANA,gene,66504959,66550130,NaN,+,0,ENSG00000174442.13,protein_coding,...,,,HGNC:25468,OTTHUMG00000133194.8,,,,,66504959,45.171


In [7]:
from src.data_preprocess import process_gtf_info, get_peak_coords
from tqdm import tqdm

print(f"\n--- 开始计算 RP Score (Decay={decay_dist}, Range={max_range}) ---")


valid_genes = set(adata_rna.var_names)
gene_df = process_gtf_info(gene_info_df, valid_genes)
common_genes = sorted(list(set(gene_df.index) & set(adata_rna.var_names)))
gene_df = gene_df.loc[common_genes]

peaks_df = get_peak_coords(adata_atac)

# 映射索引
peak_to_idx = {peak: i for i, peak in enumerate(peaks_df.index)}
gene_to_idx = {gene: i for i, gene in enumerate(common_genes)}

# 结果容器
final_rows = []
final_cols = []
final_data = []

chromosomes = gene_df['seqname'].unique()


--- 开始计算 RP Score (Decay=50000, Range=250000) ---
--- 1. 清洗 Gene Info ---


In [18]:
gene_info_df

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_type,...,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
7151,chr1,HAVANA,gene,631074,632616,NaN,+,0,ENSG00000237973.1,unprocessed_pseudogene,...,,,,,HGNC:52014,OTTHUMG00000002333.2,,,,
8830,chr1,HAVANA,gene,943527,960714,NaN,-,0,ENSG00000188976.12,protein_coding,...,,,,,HGNC:24517,OTTHUMG00000040720.2,,,,
11569,chr1,HAVANA,gene,960576,965719,NaN,+,0,ENSG00000187961.16,protein_coding,...,,,,,HGNC:24023,OTTHUMG00000040721.8,,,,
17275,chr1,HAVANA,gene,1311585,1324687,NaN,-,0,ENSG00000127054.23,protein_coding,...,,,,,HGNC:26052,OTTHUMG00000003330.13,,,,
18273,chr1,HAVANA,gene,1324756,1328897,NaN,+,0,ENSG00000224051.8,protein_coding,...,,,,,HGNC:28116,OTTHUMG00000003171.4,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7723202,chrX,HAVANA,gene,154458280,154477779,NaN,+,0,ENSG00000130827.7,protein_coding,...,,,,,HGNC:9101,OTTHUMG00000033290.6,,,,
7728601,chrX,HAVANA,gene,155197007,155239841,NaN,+,0,ENSG00000155959.13,protein_coding,...,,,,,HGNC:12662,OTTHUMG00000022666.5,,,,
7729478,chrX,HAVANA,gene,155881227,155943769,NaN,+,0,ENSG00000124333.17,protein_coding,...,,,,,HGNC:11486,OTTHUMG00000022679.3,,,,
7730016,chrY,HAVANA,gene,276322,303358,NaN,+,0,ENSG00000292344.2,protein_coding,...,,,,,HGNC:23148,OTTHUMG00000022693.6,,,,


In [24]:
from scipy import sparse
gene_df = gene_df[["seqname", "tss", "start", "end", "gene_length"]].copy()
for chrom in tqdm(chromosomes, desc="Chromosomes"):
    # 获取当前染色体的数据
    sub_genes = gene_df[gene_df['seqname'] == chrom]
    sub_peaks = peaks_df[peaks_df['chrom'] == chrom]
    
    if sub_genes.empty or sub_peaks.empty:
        continue
        
    # === 向量化计算 ===
    # Genes: (N, 1)
    g_tss = sub_genes['tss'].values.reshape(-1, 1)
    g_start = sub_genes['start'].values.reshape(-1, 1)
    g_end = sub_genes['end'].values.reshape(-1, 1)
    g_len = sub_genes['gene_length'].values.reshape(-1, 1)
    
    # Peaks: (1, M)
    p_centers = sub_peaks['center'].values.reshape(1, -1)
    
    # 1. 计算绝对距离
    dists = np.abs(p_centers - g_tss)
    
    mask_valid = dists <= max_range
    
    scores = np.power(2.0, -(dists / decay_dist))
    
    # 4. 处理基因内部 (Gene Body)
    # 逻辑: 如果 Peak 在基因体内，分数设为 1.0 / (基因长度 + 1)
    # 加1是为了防止基因极短导致分数爆炸
    in_gene_body = (p_centers >= g_start) & (p_centers <= g_end)
    
    # 构造 Body Score 矩阵
    body_scores = np.repeat(1.0 / g_len, p_centers.shape[1], axis=1)
    
    # 覆盖基因内部的分数
    scores[in_gene_body] = body_scores[in_gene_body]
    
    # 5. 应用 250kb 过滤器
    # 这一步通过布尔索引，只保留 <= 250kb 且分数 > 0 的点
    # 另外加一个 1e-5 的极小值过滤，保证稀疏性
    final_mask = mask_valid & (scores > 1e-5)
    
    # 提取索引
    r_idx, c_idx = np.where(final_mask)
    
    # 映射回全局索引并存储
    current_g_names = sub_genes.index.values
    current_p_names = sub_peaks.index.values
    
    # 这里为了速度，不使用 append，改用 extend
    # 我们需要先获取 sub_genes 在全局 common_genes 中的索引
    g_global_base_indices = [gene_to_idx[g] for g in current_g_names]
    p_global_base_indices = [peak_to_idx[p] for p in current_p_names]
    
    # 利用 numpy 的高级索引快速映射
    g_global_mapped = np.array(g_global_base_indices)[r_idx]
    p_global_mapped = np.array(p_global_base_indices)[c_idx]
    score_values = scores[r_idx, c_idx]
    
    final_rows.extend(g_global_mapped)
    final_cols.extend(p_global_mapped)
    final_data.extend(score_values)

print("--- 3. 构建 AnnData 对象 ---")

rp_matrix = sparse.csr_matrix(
    (final_data, (final_rows, final_cols)), 
    shape=(len(common_genes), len(peaks_df))
)

adata_rp = ad.AnnData(
    X=rp_matrix,
    obs=pd.DataFrame(index=common_genes),
    var=pd.DataFrame(index=peaks_df.index)
)

# 记录参数到 uns
adata_rp.uns['decay_distance'] = decay_dist
adata_rp.uns['max_range'] = max_range
adata_rp.uns['description'] = f"RP Score (Decay={decay_dist/1000}kb, Max={max_range/1000}kb)"

print(f"✅ 完成! 矩阵维度: {adata_rp.shape}")
print(f"非零连接数: {rp_matrix.nnz}")



Chromosomes: 100%|██████████| 23/23 [00:00<00:00, 44.57it/s]

--- 3. 构建 AnnData 对象 ---
✅ 完成! 矩阵维度: (2000, 72584)
非零连接数: 49252


In [25]:
from src.data_preprocess import adata_to_dataframe
rp_df = adata_to_dataframe(adata_rp)

In [26]:
rp_df

,obs,var,value
0,AAAS,chr12-53078932-53079727,0.099736
1,AAAS,chr12-53097571-53098404,0.129177
2,AAAS,chr12-53099350-53099648,0.131913
3,AAAS,chr12-53102978-53104124,0.139535
4,AAAS,chr12-53113109-53114214,0.160529
...,...,...,...
49247,ZYX,chr7-143327917-143328230,0.956316
49248,ZYX,chr7-143362369-143362688,1.541847
49249,ZYX,chr7-143380273-143381719,1.991727
49250,ZYX,chr7-143408952-143409231,1.360446


In [27]:
adata_rp = ad.read_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")

In [28]:
rp_df = adata_to_dataframe(adata_rp)
print(rp_df)

        obs                       var  value
0      AAAS   chr12-53251618-53252739      1
1      AAAS   chr12-53267768-53268827      1
2      AAAS   chr12-53295185-53295894      1
3      AAAS   chr12-53299560-53300133      1
4      AAAS   chr12-53336297-53336656      1
...     ...                       ...    ...
12237  ZXDC  chr3-126522381-126522675      1
12238   ZYX  chr7-143327917-143328230      1
12239   ZYX  chr7-143362369-143362688      1
12240   ZYX  chr7-143380273-143381719      1
12241   ZYX  chr7-143408952-143409231      1

[12242 rows x 3 columns]


In [30]:
adata_rp_gene_peak = ad.read_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")
adata_rp_peak = ad.read_h5ad(output_path + "binary_peak_peak_rp_network.h5ad")

# **************************** Preprocess Data *******************************
# ****************************  统一基因和peak
print("******************* Consistency Gene ***************************")
print(f"Before filter: adata_rna have gene: {adata_rna.shape[1]}, adata_rp_gene_preak have gene: {adata_rp_gene_peak.shape[0]}")

total_gene= set(adata_rna.var_names) & set(adata_rp_gene_peak.obs_names)
adata_rna = adata_rna[:,list(total_gene)].copy()
adata_rp_gene_peak = adata_rp_gene_peak[list(total_gene),:].copy()

print(f"After filter: adata_rna have gene: {adata_rna.shape[1]}, adata_rp_gene_preak have gene: {adata_rp_gene_peak.shape[0]}")


print("******************* Consistency Peak ***************************")

print(f"Before filter: adata_atac have peak: {adata_atac.shape[1]}")
print(f"adata_rp_gene_peak have peak: {adata_rp_gene_peak.shape[0]}")
print(f"adata_rp_peak have peak: {adata_rp_peak.shape[1]}")

total_peak= set(adata_atac.var_names) & set(adata_rp_gene_peak.var_names) & set(adata_rp_peak.var_names)
adata_atac = adata_atac[:,list(total_peak)].copy()
adata_rp_gene_peak = adata_rp_gene_peak[:, list(total_peak)].copy()

******************* Consistency Gene ***************************
Before filter: adata_rna have gene: 2000, adata_rp_gene_preak have gene: 2000
After filter: adata_rna have gene: 2000, adata_rp_gene_preak have gene: 2000
******************* Consistency Peak ***************************
Before filter: adata_atac have peak: 72584
adata_rp_gene_peak have peak: 2000
adata_rp_peak have peak: 71541


In [31]:
rp_df2 = adata_to_dataframe(adata_rp_gene_peak)
print(rp_df2)

           obs                       var  value
0      FAM168B  chr2-131104645-131106587      1
1      FAM168B  chr2-131144994-131145579      1
2      FAM168B  chr2-131153661-131154556      1
3        FBXL5    chr4-15681319-15682269      1
4        FBXL5    chr4-15729880-15730525      1
...        ...                       ...    ...
12237   YTHDF1   chr20-63223198-63223572      1
12238   YTHDF1   chr20-63230739-63231065      1
12239   YTHDF1   chr20-63259698-63260029      1
12240   YTHDF1   chr20-63272174-63273301      1
12241   YTHDF1   chr20-63290843-63291518      1

[12242 rows x 3 columns]


In [35]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result/"
adata_rp_gene_peak = ad.read_h5ad(output_path + "rp_gene_peak.h5ad")

rp_df3 = adata_to_dataframe(adata_rp_gene_peak)

print(rp_df3)

         obs                       var     value
0      DHX36  chr3-154121259-154122063  0.060099
1      DHX36  chr3-154252922-154253458  0.372177
2      DHX36  chr3-154304457-154305305  0.019253
3      DHX36  chr3-154323800-154324741  0.019253
4      DHX36  chr3-154398041-154398856  0.358683
...      ...                       ...       ...
48961   SVIP   chr11-22786416-22787795  0.549473
48962   SVIP   chr11-22807904-22808739  0.737359
48963   SVIP   chr11-22829271-22830179  0.060606
48964   SVIP   chr11-22930948-22931542  0.246743
48965   SVIP   chr11-23078454-23079791  0.031764

[48966 rows x 3 columns]


# Other

In [ ]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"
adata_atac = ad.read_h5ad(output_path + "atac_processed.h5ad")
adata_rna = ad.read_h5ad(output_path + "rna_processed.h5ad")
gene_info = pd.read_pickle(output_path + "gene_info_filtered.pkl")
peak_gene_grn = ad.read_h5ad(output_path + "peak_gene_network.h5ad")

In [43]:
adata_rp = ad.read_h5ad(output_path + "peak_gene_rp_network.h5ad")

adata_peak_rp = ad.read_h5ad(output_path + "peak_peak_rp_network.h5ad")

binary_adata_rp = ad.read_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")

binary_adata_peak_rp = ad.read_h5ad(output_path + "binary_peak_peak_rp_network.h5ad")


In [44]:
print(f"非零连接数: {adata_rp.X.nnz}, Sparsity: {100 * adata_rp.X.nnz / (adata_rp.X.shape[0] * adata_rp.shape[1]):.4f}%")
print(f"非零连接数: {adata_peak_rp.X.nnz}, Sparsity: {100 * adata_peak_rp.X.nnz / (adata_peak_rp.X.shape[0] * adata_peak_rp.shape[1]):.4f}%")
print(f"非零连接数: {binary_adata_rp.X.nnz}, Sparsity: {100 * binary_adata_rp.X.nnz / (binary_adata_rp.X.shape[0] * binary_adata_rp.shape[1]):.4f}%")
print(f"非零连接数: {binary_adata_peak_rp.X.nnz}, Sparsity: {100 * binary_adata_peak_rp.X.nnz / (binary_adata_peak_rp.X.shape[0] * binary_adata_peak_rp.shape[1]):.4f}%")

非零连接数: 50061, Sparsity: 0.0793%
非零连接数: 783625, Sparsity: 0.0787%
非零连接数: 12515, Sparsity: 0.0198%
非零连接数: 195903, Sparsity: 0.0197%


In [47]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

# 1. 把所有数据放入字典，方便批量处理
adatas = {
    "adata_rp": adata_rp,
    "adata_peak_rp": adata_peak_rp,
    "binary_adata_rp": binary_adata_rp,
    "binary_adata_peak_rp": binary_adata_peak_rp
}

summary_list = []

print(f"{'Dataset':<25} | {'Shape (Rows x Cols)':<20} | {'Avg Row Non-Zeros':<20} | {'Avg Col Non-Zeros':<20}")
print("-" * 95)

for name, adata in adatas.items():
    
    if sparse.issparse(adata.X):
        row_counts = adata.X.getnnz(axis=1)
    else:
        row_counts = np.count_nonzero(adata.X, axis=1)
        
    # 找出非零的行索引 (即 count > 0 的行)
    # 这就是您要的“记录哪些行的和是非零的”
    active_row_mask = row_counts > 0
    active_rows = adata.obs_names[active_row_mask].tolist()
    
    print(f"  - 行总数: {adata.n_obs}")
    print(f"  - 非零行数 (有连接的行): {len(active_rows)}")
    print(f"  - 全零行数 (无连接的行): {adata.n_obs - len(active_rows)}")

    # ---------------------------
    # 2. 分析列 (Var / Cols)
    # ---------------------------
    # 计算每一列的非零个数
    if sparse.issparse(adata.X):
        col_counts = adata.X.getnnz(axis=0)
    else:
        col_counts = np.count_nonzero(adata.X, axis=0)
        
    # 找出非零的列索引
    active_col_mask = col_counts > 0
    active_cols = adata.var_names[active_col_mask].tolist()
    
    
    # 记录详细统计供后续查看
    summary_list.append({
        "Dataset": name,
        "Row_Total": adata.n_obs,
        "Row_Num": len(active_rows),
        "Col_Total": adata.n_vars,
        "Col_NNZ": len(active_cols),
        "Num links": adata.X.nnz, 
        "Sparsity": 100 * adata.X.nnz / (adata.X.shape[0] * adata.shape[1])
    })

# 如果需要详细表格
df_summary = pd.DataFrame(summary_list)

print(df_summary)

Dataset                   | Shape (Rows x Cols)  | Avg Row Non-Zeros    | Avg Col Non-Zeros   
-----------------------------------------------------------------------------------------------
  - 行总数: 2000
  - 非零行数 (有连接的行): 1999
  - 全零行数 (无连接的行): 1
  - 行总数: 31555
  - 非零行数 (有连接的行): 31555
  - 全零行数 (无连接的行): 0
  - 行总数: 2000
  - 非零行数 (有连接的行): 1905
  - 全零行数 (无连接的行): 95
  - 行总数: 31555
  - 非零行数 (有连接的行): 31555
  - 全零行数 (无连接的行): 0
                Dataset  Row_Total  Row_Num  Col_Total  Col_NNZ  Num links  \
0              adata_rp       2000     1999      31555    30851      50061   
1         adata_peak_rp      31555    31555      31555    31555     783625   
2       binary_adata_rp       2000     1905      31555    10348      12515   
3  binary_adata_peak_rp      31555    31555      31555    31555     195903   

   Sparsity  
0  0.079323  
1  0.078699  
2  0.019830  
3  0.019675  
